In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "MCQ_C.xpt"
csv_file = DATA_DIR / "MCQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for fidelity checking
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 MCQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. BASIC STRUCTURE CHECK
# ============================================================

if len(df) != 9645:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )

if len(df.columns) != 154:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


REQUIRED_COLUMNS = [
    "SEQN",
    "MCQ010",
    "MCQ150Q",
    "MCQ180K",
    "MCQ180M",
    "MCQ220",
    "MCQ245B"
]


missing_columns = [
    col
    for col in REQUIRED_COLUMNS
    if col not in df.columns
]


if missing_columns:
    raise ValueError(
        "Required MCQ columns missing:\n"
        + str(missing_columns)
    )


print(
    "PASS: MCQ_C structure = "
    "9,645 rows x 154 columns."
)


# ============================================================
# 4. KNOWN PANDAS/XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. ONLY THESE VARIABLES MAY HAVE ZERO RESTORED
# ============================================================

#
# CDC supports zero for:
#
# MCQ150Q = school days missed
# MCQ245B = work days missed
# MCQ180K = age diagnosed chronic bronchitis
# MCQ180M = age diagnosed thyroid problem
#

ZERO_ALLOWED_COLUMNS = [
    "MCQ150Q",
    "MCQ245B",
    "MCQ180K",
    "MCQ180M"
]


EXPECTED_TINY_COUNTS = {
    "MCQ150Q": 1000,
    "MCQ245B": 1954,
    "MCQ180K": 3,
    "MCQ180M": 1
}


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

allowed_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    allowed_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values appeared in unexpected "
        "MCQ variables:\n"
        + str(sorted(unexpected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts.get(
            col,
            0
        )
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} "
            f"tiny values but found {actual_count}"
        )


print(
    "\nPASS: All 2,958 tiny values occur only "
    "in CDC zero-permitted fields."
)


# ============================================================
# 7. RESTORE TRUE ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = df[col].eq(
        TINY_VALUE
    )

    count = int(
        mask.sum()
    )

    print(
        f"{col}: {count:,} tiny values -> 0"
    )

    df.loc[
        mask,
        col
    ] = 0.0


# ============================================================
# 8. VERIFY RESTORED ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )

    print(
        f"{col}: {actual_zero:,}"
    )

    if actual_zero != expected_count:

        raise ValueError(
            f"{col}: zero count differs after restoration."
        )


print(
    "PASS: All expected zeros restored."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# Never round real decimals.
#
# First check the entire XPT.
# If ANY genuine fractional value exists,
# stop before converting to integer formatting.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional = ~np.isclose(
        values,
        np.round(values)
    )


    if fractional.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nReal decimal values found in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values exist. "
        "These columns will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in this MCQ_C XPT."
)


# ============================================================
# 10. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 11. FORMAT ALL MCQ FIELDS
# ============================================================

#
# We verified above that EVERY released MCQ value
# in this XPT is whole-number.
#
# Therefore:
#
# 1.0     -> 1
# 65.0    -> 65
# 99999.0 -> 99999
# 0.0     -> 0
#
# Missing remains missing.
#
# We do NOT use round() here.
#

MCQ_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in MCQ_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: MCQ codes/counts/ages formatted "
    "without unnecessary .0."
)


# ============================================================
# 12. CDC CHECK - MCQ010
# ============================================================

#
# Ever told you have asthma
#
# CDC:
# 1 = Yes          1336
# 2 = No           8290
# 9 = Don't know     19
# Missing              0
#

EXPECTED_MCQ010 = {
    1: 1336,
    2: 8290,
    9: 19
}


actual = (
    df["MCQ010"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- MCQ010 ---")


if (
    actual == EXPECTED_MCQ010
    and
    int(df["MCQ010"].isna().sum()) == 0
):

    print(
        "PASS: MCQ010 matches CDC."
    )

else:

    raise ValueError(
        "MCQ010 differs from CDC."
    )


# ============================================================
# 13. CDC CHECK - MCQ150Q
# ============================================================

#
# CDC:
#
# 0-150 = 3098
# 99999 = 5
# Missing = 6542
#

s = df["MCQ150Q"]


regular = int(
    s.between(
        0,
        150
    ).sum()
)

dont_know = int(
    s.eq(99999).sum()
)

refused = int(
    s.eq(77777).sum()
)

missing = int(
    s.isna().sum()
)


print("\n--- MCQ150Q ---")

print("0-150:", regular)
print("Zeros:", int(s.eq(0).sum()))
print("77777:", refused)
print("99999:", dont_know)
print("Missing:", missing)


if (
    regular == 3098
    and
    refused == 0
    and
    dont_know == 5
    and
    missing == 6542
):

    print(
        "PASS: MCQ150Q matches CDC."
    )

else:

    raise ValueError(
        "MCQ150Q differs from CDC."
    )


# ============================================================
# 14. CDC CHECK - MCQ180K
# ============================================================

#
# CDC:
#
# 0-83   = 323
# 85     = 1
# 77777  = 0
# 99999  = 7
# Missing = 9314
#

s = df["MCQ180K"]


regular = int(
    s.between(
        0,
        83
    ).sum()
)


print("\n--- MCQ180K ---")

print("0-83:", regular)
print("Zeros:", int(s.eq(0).sum()))
print("85:", int(s.eq(85).sum()))
print("99999:", int(s.eq(99999).sum()))
print("Missing:", int(s.isna().sum()))


if (
    regular == 323
    and
    int(s.eq(85).sum()) == 1
    and
    int(s.eq(77777).sum()) == 0
    and
    int(s.eq(99999).sum()) == 7
    and
    int(s.isna().sum()) == 9314
):

    print(
        "PASS: MCQ180K matches CDC."
    )

else:

    raise ValueError(
        "MCQ180K differs from CDC."
    )


# ============================================================
# 15. CDC CHECK - MCQ180M
# ============================================================

#
# CDC:
#
# 0-84   = 485
# 85     = 8
# 77777  = 0
# 99999  = 5
# Missing = 9147
#

s = df["MCQ180M"]


regular = int(
    s.between(
        0,
        84
    ).sum()
)


print("\n--- MCQ180M ---")

print("0-84:", regular)
print("Zeros:", int(s.eq(0).sum()))
print("85:", int(s.eq(85).sum()))
print("99999:", int(s.eq(99999).sum()))
print("Missing:", int(s.isna().sum()))


if (
    regular == 485
    and
    int(s.eq(85).sum()) == 8
    and
    int(s.eq(77777).sum()) == 0
    and
    int(s.eq(99999).sum()) == 5
    and
    int(s.isna().sum()) == 9147
):

    print(
        "PASS: MCQ180M matches CDC."
    )

else:

    raise ValueError(
        "MCQ180M differs from CDC."
    )


# ============================================================
# 16. CDC CHECK - MCQ220
# ============================================================

#
# Ever told you had cancer or malignancy
#
# CDC:
#
# 1 = Yes          478
# 2 = No          4551
# 9 = Don't know    11
# Missing          4605
#

EXPECTED_MCQ220 = {
    1: 478,
    2: 4551,
    9: 11
}


actual = (
    df["MCQ220"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- MCQ220 ---")


if (
    actual == EXPECTED_MCQ220
    and
    int(df["MCQ220"].isna().sum()) == 4605
):

    print(
        "PASS: MCQ220 matches CDC."
    )

else:

    raise ValueError(
        "MCQ220 differs from CDC."
    )


# ============================================================
# 17. CDC CHECK - MCQ245B
# ============================================================

#
# CDC:
#
# 0-365  = 3553
# 99999  = 3
# Missing = 6089
#

s = df["MCQ245B"]


regular = int(
    s.between(
        0,
        365
    ).sum()
)


print("\n--- MCQ245B ---")

print("0-365:", regular)
print("Zeros:", int(s.eq(0).sum()))
print("77777:", int(s.eq(77777).sum()))
print("99999:", int(s.eq(99999).sum()))
print("Missing:", int(s.isna().sum()))


if (
    regular == 3553
    and
    int(s.eq(77777).sum()) == 0
    and
    int(s.eq(99999).sum()) == 3
    and
    int(s.isna().sum()) == 6089
):

    print(
        "PASS: MCQ245B matches CDC."
    )

else:

    raise ValueError(
        "MCQ245B differs from CDC."
    )


# ============================================================
# 18. BUILD EXPECTED SOURCE
# ============================================================

#
# Expected source = original XPT
# with ONLY four validated zero restorations.
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 19. VERIFY EVERY VALUE BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(failures)
    )


print(
    "PASS: All 154 variables preserved exactly "
    "after validated zero restoration."
)


# ============================================================
# 20. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO CHECK ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny artifacts still remain."
    )


print(
    "PASS: No tiny-value artifact remains."
)


# ============================================================
# 21. EXPORT CSV
# ============================================================

#
# NO float_format.
# NO recalculation.
# NO arbitrary rounding.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 22. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 23. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 154 columns match expected XPT values."
)


# ============================================================
# 24. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 25. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


print("\n" + "=" * 75)
print("MCQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 MCQ_C XPT -> CSV
Rows: 9,645
Columns: 154
PASS: MCQ_C structure = 9,645 rows x 154 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 2,958

Tiny values by variable:
MCQ245B    1954
MCQ150Q    1000
MCQ180K       3
MCQ180M       1

PASS: All 2,958 tiny values occur only in CDC zero-permitted fields.

--- ZERO RESTORATION ---
MCQ150Q: 1,000 tiny values -> 0
MCQ245B: 1,954 tiny values -> 0
MCQ180K: 3 tiny values -> 0
MCQ180M: 1 tiny values -> 0

--- RESTORED ZERO COUNTS ---
MCQ150Q: 1,000
MCQ245B: 1,954
MCQ180K: 3
MCQ180M: 1
PASS: All expected zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in this MCQ_C XPT.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126

PASS: MCQ codes/counts/ages formatted without unnecessary .0.

--- MCQ010 ---
PASS: MCQ010 matches CDC.

--- MCQ150Q ---
0-150: 3098
Zeros: 1000
77777: 0
99999: 5
Missing: 6542
PASS: MCQ150Q matches CDC.

--- MCQ180K -